# 04 · Final predictions and benchmark evidence

**Read the saved results. No kernel, AWS account, or data download is needed.**

This notebook connects the current model comparison to actual final fits and a real 2026 prediction file. The procedure freezes candidates from 2016–2019 and 2021 development forecasts, evaluates the already-consumed 2022–2025 seasons, and refits through 2025. **No 2026 tournament outcomes enter selection, calibration, or fitting.**

The official metric is **Brier score** (mean squared probability error; lower is better). Our historical game population includes play-ins, unlike Kaggle's scored 2026 set. These retrospective benchmark values are **not** leaderboard scores or untouched-holdout estimates. The original competitive deadline has passed.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.runtime import digest

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
FINAL = ROOT / "reports/final_predictions"
style()
ready = (FINAL / "evidence.json").is_file()
if ready:
    evidence = json.loads((FINAL / "evidence.json").read_text())
    for name, expected in evidence["sha256"].items():
        path = FINAL / name
        assert path.resolve().is_relative_to(FINAL.resolve())
        assert digest(path) == expected, name
    summary = json.loads((FINAL / "summary.json").read_text())
    assert summary["status"] == "completed" and not summary["future_labels_used"]
    display(Markdown(f"**{summary['rows']:,} real matchup probabilities · "
                     f"{summary['final_model_fits']} final estimators · "
                     f"training through {summary['final_training_max_season']}**"))
    display(Markdown("Submission SHA-256: `" + summary["submission_sha256"] + "`"))
else:
    display(Markdown("Current final-run evidence has not been published in this checkout. "
                     "Historical records below are not substituted for it."))

## Frozen recipe and complete matchup coverage

Men use the development-selected ranking-logistic family; women use the development-selected separate logistic family. Within each family, the feature block and regularization are selected from saved development out-of-fold predictions using mean season Brier. Identity calibration is retained; nothing is tuned on the displayed benchmark.

When both teams have tournament seeds, the seeded estimator is used. Otherwise, a separately fitted **seed-free estimator removes every seed-dependent input**. Missing seeds are not invented. All template rows retain the required lower-TeamID win-probability orientation.

In [ ]:
if ready:
    recipe = json.loads((FINAL / "recipe.json").read_text())
    fits = json.loads((FINAL / "fit_audits.json").read_text())
    table(pd.DataFrame([{"Tournament": r["gender"], "Route": r["route"],
                         "Candidate": r["candidate"]["name"],
                         "Features": len(r["features"]), "Training games": r["training_games"],
                         "Last training season": r["training_max_season"]} for r in fits]))
    table(pd.read_csv(FINAL / "routes.csv"))

## Retrospective 2022–2025 benchmark

Each season is predicted using only earlier seasons, with the same frozen recipe. The seed-free route is also evaluated on the same tournament games to expose the value of seed information. Its tournament performance does not prove generalization to teams that never qualified.

**Game-weighted Brier** averages every game's squared error. **Mean season Brier** weights seasons equally. Log loss, ROC AUC, average precision and calibration error provide complementary diagnostics. The benchmark was consumed in prior work and is not used to replace the frozen recipe.

In [ ]:
if ready:
    current = pd.read_csv(FINAL / "metrics.csv")
    seasonal = pd.read_csv(FINAL / "metrics_by_season.csv")
    table(current[["Gender", "route", "games", "brier", "mean_season_brier",
                   "log_loss", "roc_auc", "average_precision", "ece_10_bins"]])
    fig, ax = plt.subplots(figsize=(9, 4.6), constrained_layout=True)
    for (gender, route), group in seasonal.groupby(["Gender", "route"]):
        group = group.sort_values("Season")
        ax.plot(group.Season, group.brier, marker="o",
                linestyle="-" if route == "seeded" else "--",
                label=f"{'Men' if gender == 'M' else 'Women'} · {route.replace('_',' ')}")
    ax.set(title="Frozen recipe across previously consumed seasons",
           xlabel="Predicted tournament season", ylabel="Brier score · lower is better",
           xticks=[2022, 2023, 2024, 2025])
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=.2)
    ax.legend(frameon=False, ncol=2, fontsize=9)
    plt.show()

In [ ]:
if ready:
    reliability = pd.read_csv(FINAL / "reliability.csv")
    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    ax.plot([0,1], [0,1], linestyle="--", alpha=.5, label="Perfect calibration")
    for gender, group in reliability.loc[reliability.route.eq("seeded")].groupby("Gender"):
        ax.plot(group.predicted, group.observed, marker="o",
                label=f"{'Men' if gender == 'M' else 'Women'} · seeded")
    ax.set(title="Calibration · retrospective seeded tournament forecasts",
           xlabel="Mean predicted win probability", ylabel="Observed win frequency",
           xlim=(0,1), ylim=(0,1))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=.15)
    ax.legend(frameon=False)
    plt.show()
    display(Markdown("Points summarize ten probability bins. Small bin counts and only four "
                     "seasons limit precision; proximity to the diagonal is not proof of "
                     "future calibration. No curve here is used for post-benchmark tuning."))

## Durability and release status

Each estimator and prediction batch is checkpointed only after output hashes are recorded. The private versioned S3 copy supports recovery independently of the notebook kernel or runner. The verification below starts in a fresh local run directory and requires identical CSV bytes with **zero repeated fits**.

The prediction CSV is an actual generated deliverable, but generation is not a Kaggle upload. The release records that distinction explicitly.

In [ ]:
if ready:
    recovery = json.loads((FINAL / "resume.json").read_text())
    assert recovery["status"] == "passed" and recovery["repeated_fits"] == 0
    table(pd.DataFrame([{"Recovery": recovery["status"], "Tasks restored": recovery["restored_tasks"],
                         "Tasks reused": recovery["reused_tasks"], "Fits repeated": recovery["repeated_fits"],
                         "Identical CSV": recovery["identical_submission"],
                         "Kaggle upload sent": summary["kaggle_submission_sent"]}]))
    display(Markdown("**Current final-fit review complete.** The linked historical reports preserve "
                     "the older model lineage; its scores and models are not mixed with the current run."))

## Reproduction and sources

Maintainers run the **Final predictions** GitHub workflow or `python -m march_mania.publication.inference --download-inputs --s3 <private-prefix>`. Readers need neither. The pipeline verifies the existing input archives, reuses successful checkpoints, and never repeats the completed 497-fit comparison just to publish a final CSV.

`scripts/portfolio_release.py` remains the independent exact-CSV auditing command. The original competitive deadline was **March 19, 2026, 16:00 UTC**. Later predictions are retrospective; a new Kaggle score requires an accepted upload and its actual receipt.

Sources: [official evaluation and timeline](https://www.kaggle.com/competitions/march-machine-learning-mania-2026) · [organizer's scored-game clarification](https://www.kaggle.com/competitions/march-machine-learning-mania-2026/discussion/680419) · current `reports/final_predictions/` · historical `reports/final_2026/` and `reports/submission_portfolio/`.

The earlier model lineage remains in [the historical final-model report](../reports/final_2026/) and [submission score log](../reports/submission_portfolio/). Historical neural and margin results are not relabeled as current-schema reruns.